In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Libraries imported successfully!")

Libraries imported successfully!


In [6]:
df = pd.read_csv(
    "../data/raw/synthetic_atm_data.csv",
    parse_dates=["date"]
)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (6036, 21)


,zone_id,date,latitude,longitude,day_of_week,month,is_weekend,atm_count,withdrawal_count,avg_withdrawal_amount,...,unusual_activity_count,cashout_incident_count,withdrawals_sum_7d,unusual_activity_sum_7d,incidents_sum_7d,withdrawals_sum_30d,unusual_activity_sum_30d,incidents_sum_30d,target_incidents_next_7d,target_risk
0,ZONE_01,2025-01-31,31.000914,75.000198,4,1,0,9,121,1953.02,...,3,0,621.0,8.0,4.0,2816.0,42.0,10.0,2,Medium
1,ZONE_01,2025-02-01,31.000914,75.000198,5,2,1,4,113,1508.52,...,2,0,647.0,11.0,3.0,2841.0,45.0,9.0,2,Medium
2,ZONE_01,2025-02-02,31.000914,75.000198,6,2,1,4,104,2560.12,...,1,0,668.0,12.0,2.0,2854.0,47.0,9.0,3,Medium
3,ZONE_01,2025-02-03,31.000914,75.000198,0,2,0,9,104,4288.27,...,1,1,693.0,13.0,2.0,2851.0,45.0,9.0,2,Medium
4,ZONE_01,2025-02-04,31.000914,75.000198,1,2,0,12,77,2664.90,...,0,1,719.0,12.0,3.0,2851.0,44.0,10.0,1,Low


In [7]:
missing_values = df.isnull().sum()

print("Missing Values:")
print(missing_values[missing_values > 0])

print("\nTotal Missing Values:", df.isnull().sum().sum())

Missing Values:
Series([], dtype: int64)

Total Missing Values: 0


In [8]:
numeric_cols = [
    "withdrawal_count",
    "avg_withdrawal_amount",
    "total_withdrawal_amount",
    "unusual_activity_count",
    "incidents_sum_7d",
    "incidents_sum_30d"
]

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) | (df[col] > upper)
    ]

    print(f"{col}: {len(outliers)} potential outliers")

withdrawal_count: 19 potential outliers
avg_withdrawal_amount: 36 potential outliers
total_withdrawal_amount: 74 potential outliers
unusual_activity_count: 73 potential outliers
incidents_sum_7d: 76 potential outliers
incidents_sum_30d: 118 potential outliers


In [9]:
df["withdrawals_per_atm"] = (
    df["withdrawal_count"] / df["atm_count"]
)

df["unusual_activity_ratio"] = (
    df["unusual_activity_count"] /
    df["withdrawal_count"].replace(0, np.nan)
)

df["avg_daily_incidents_30d"] = (
    df["incidents_sum_30d"] / 30
)

df["withdrawal_amount_per_atm"] = (
    df["total_withdrawal_amount"] / df["atm_count"]
)

df[[
    "withdrawals_per_atm",
    "unusual_activity_ratio",
    "avg_daily_incidents_30d",
    "withdrawal_amount_per_atm"
]].head()

,withdrawals_per_atm,unusual_activity_ratio,avg_daily_incidents_30d,withdrawal_amount_per_atm
0,13.444444,0.024793,0.333333,26257.272222
1,28.250000,0.017699,0.300000,42615.680000
2,26.000000,0.009615,0.300000,66563.170000
3,11.555556,0.009615,0.300000,49553.321111
4,6.416667,0.000000,0.333333,17099.747500


In [10]:
features = [
    "zone_id",
    "latitude",
    "longitude",
    "day_of_week",
    "month",
    "is_weekend",
    "atm_count",
    "withdrawal_count",
    "avg_withdrawal_amount",
    "total_withdrawal_amount",
    "unusual_activity_count",
    "withdrawals_sum_7d",
    "unusual_activity_sum_7d",
    "incidents_sum_7d",
    "withdrawals_sum_30d",
    "unusual_activity_sum_30d",
    "incidents_sum_30d",
    "withdrawals_per_atm",
    "unusual_activity_ratio",
    "avg_daily_incidents_30d",
    "withdrawal_amount_per_atm"
]

X = df[features]

y = df["target_risk"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (6036, 21)
Target: (6036,)


In [11]:
df = df.sort_values("date").reset_index(drop=True)

unique_dates = np.sort(df["date"].unique())

split_index = int(len(unique_dates) * 0.80)

test_start = pd.Timestamp(unique_dates[split_index])

# Leave a 7-day gap to prevent overlapping forecast targets
train_end = test_start - pd.Timedelta(days=7)

train_mask = df["date"] < train_end
test_mask = df["date"] >= test_start

X_train = df.loc[train_mask, features]
X_test = df.loc[test_mask, features]

y_train = df.loc[train_mask, "target_risk"]
y_test = df.loc[test_mask, "target_risk"]

print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))

print("\nTraining Period:")
print(df.loc[train_mask, "date"].min(), "to",
      df.loc[train_mask, "date"].max())

print("\nTesting Period:")
print(df.loc[test_mask, "date"].min(), "to",
      df.loc[test_mask, "date"].max())

Training Records: 4740
Testing Records: 1212

Training Period:
2025-01-31 00:00:00 to 2026-03-01 00:00:00

Testing Period:
2026-03-09 00:00:00 to 2026-06-17 00:00:00


In [12]:
numeric_features = [
    col for col in features
    if col != "zone_id"
]

categorical_features = ["zone_id"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [14]:
# Validate preprocessing on training data only

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original testing shape:", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

Original training shape: (4740, 21)
Processed training shape: (4740, 32)
Original testing shape: (1212, 21)
Processed testing shape: (1212, 32)
